# Intent-context probes: Colab driver

This notebook supplies GPU compute and persistent artifact storage. The versioned implementation lives in `src/` and `scripts/`. Run cells in order. Milestone 1 stops after the Qwen3-4B smoke test.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os

drive.mount('/content/drive')
persistent_root = Path('/content/drive/MyDrive/MATS_INTENT_PROBE/artifacts')
persistent_root.mkdir(parents=True, exist_ok=True)
os.environ['MATS_PERSISTENT_ARTIFACT_ROOT'] = str(persistent_root)

github_token = userdata.get('GH_TOKEN')
assert github_token, 'Add GH_TOKEN in the Colab Secrets panel.'
os.environ['GH_TOKEN'] = github_token
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
print('Persistent artifact root:', persistent_root)

In [ ]:
!apt-get update -qq
!apt-get install -y gh
!gh auth status
!gh auth setup-git

In [ ]:
import subprocess
from pathlib import Path

repository = 'Oladiposamuel/intent-context-probes'
repo_path = Path('/content/intent-context-probes')
if repo_path.exists():
    subprocess.run(['git', '-C', str(repo_path), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(repo_path), 'checkout', 'master'], check=True)
    subprocess.run(['git', '-C', str(repo_path), 'pull', '--ff-only', 'origin', 'master'], check=True)
else:
    subprocess.run(['gh', 'repo', 'clone', repository, str(repo_path), '--', '--branch', 'master'], check=True)
print('Repository ready:', repo_path)

In [ ]:
%cd /content/intent-context-probes
!python -m pip install -q -U pip
!python -m pip install -q -r requirements.txt
!git rev-parse HEAD

In [ ]:
!python scripts/00_check_environment.py --config configs/experiment.yaml

If the environment check passes, run the one-model smoke test below. This downloads `Qwen/Qwen3-4B`; it does not train or modify the model.

In [ ]:
!python scripts/03_run_model.py --config configs/experiment.yaml --model qwen3_4b --smoke-test

In [ ]:
from pathlib import Path
smoke_json = Path('artifacts/smoke_tests/qwen3_4b/smoke_test.json')
persistent_json = persistent_root / 'smoke_tests/qwen3_4b/smoke_test.json'
assert smoke_json.is_file(), smoke_json
assert persistent_json.is_file(), persistent_json
print('Local artifact:', smoke_json)
print('Persistent artifact:', persistent_json)
print('STOP: return the environment and smoke-test outputs for review before bulk execution.')